# Lab 6: Covert Channels in Image Metadata
**Guided investigation | Student notebook**

**Scenario:** Six images contain pieces of a fictional cybersecurity incident narrative. Each image contains **one partial secret**. Your task is to find the six pieces, document how each was recovered, and reconstruct the full message in the correct order. **Do not execute any activity described by the messages.**

**Learning goals:** 
-  distinguish EXIF, PNG text, and XMP metadata; 
-  use short Python snippets to extract metadata;
-  recognize encoded text; 
-  record reproducible forensic evidence; 
-  assess the limitations of metadata-based conclusions.

**How this lab works:** First practise each small snippet on **one image**. Then select and run the relevant snippets on **each of the six images**, one at a time. You will write the final secret yourself. **No completed secret is provided in this student notebook.**

## 0. Prepare the environment
Keep this notebook beside the `Lab6` folder, which contains `Images`. Run the following two short cells.

In [1]:
%pip -q install Pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from PIL import Image, ExifTags

image_dir = Path("Images")
print("Images found:", len(list(image_dir.glob("*.png"))))

Images found: 6


### Your evidence log
For **each image**, record the metadata type, field name, raw value (or a short excerpt), decoding method (if any), recovered message, and your interpretation. Do not assume that `getexif()` displays all metadata.

## 1. First technique: EXIF metadata (one image)
EXIF metadata can include camera details, dates, descriptions, and other tags. Try **Brain-Channel.png** first.

In [3]:
image_path = image_dir / "Brain-Channel.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)

EXIF tags: 5
ResolutionUnit : 2
ImageDescription : LATERAL: Pivot to internal Domain Controller using Mimikatz.
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0


**Observe:** Which EXIF field carries a message? Record it in your evidence log.

**Try it yourself:** Change `image_path` to a different image. If `EXIF tags: 0` appears, does that mean the image has no metadata? Explain.

In [4]:
# Your turn: choose another image and repeat the EXIF check.
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
    # Check Sub-IFD (ExifOffset)
    sub_ifd = exif.get_ifd(ExifTags.IFD.Exif)
    print("Sub-IFD tags:", len(sub_ifd))
    for tag_id, value in sub_ifd.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)

EXIF tags: 10
GPSInfo : 396
ResolutionUnit : 2
ExifOffset : 204
Make : Nikon
Model : D750
Software : Adobe Lightroom 13.0
DateTime : 2025:02:02 10:15:00
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0
Sub-IFD tags: 7
ExifVersion : b'0232'
ComponentsConfiguration : b'\x01\x02\x03\x00'
FlashPixVersion : b'0100'
DateTimeOriginal : 2025:02:02 10:15:00
DateTimeDigitized : 2025:02:02 10:15:00
ColorSpace : 65535
UserComment : b'ASCII\x00\x00\x00VULN: Identify unpatched CVE-2024-XXXX in web gateway.'


## 2. Second technique: PNG text metadata (one image)
PNG files can contain textual information that is **not** returned by `getexif()`. Try **Brain.png**.

In [5]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

Software : RECON: Perform DNS harvesting and Subdomain enumeration.


**Observe:** Which PNG text field contains a clue? Try **Chemicals.png** next using the same short snippet.

In [6]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

ImageDescription : EXPLOIT: Deploy SQL Injection payload to bypass WAF.


## 3. Third technique: XMP metadata (one image)
XMP is XML-based metadata. It may be stored in PNG files even when EXIF is empty. Try **Dopamine.png**.

In [7]:
image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    print(xmp if xmp else "No XMP found")

<?xpacket begin='﻿' id='W5M0MpCehiHzreSzNTczkc9d'?>
<x:xmpmeta xmlns:x='adobe:ns:meta/' x:xmptk='Image::ExifTool 12.76'>
<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>

 <rdf:Description rdf:about=''
  xmlns:exif='http://ns.adobe.com/exif/1.0/'>
  <exif:MakerNote>PERSIST: Establish reverse shell via obfuscated cron job.</exif:MakerNote>
 </rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end='r'?>


**Observe:** Locate the XML element containing the partial secret. Why did the EXIF-only approach miss it? **Note:** XMP may appear under either `XML:com.adobe.xmp` or `xmp`, depending on how the file was created and the software version.

## 4. Fourth technique: Decode suspicious text (one image)
Sometimes metadata contains **encoded** rather than readable text. Try **Music.png**. First inspect its EXIF or PNG text fields. Identify a value that resembles Base64, then decode it using this small example.

In [8]:
import base64

example = "SGVsbG8="
print(base64.b64decode(example).decode("utf-8"))

Hello


**Your turn:** Replace `example` with the suspicious metadata value from **Music.png**. Do not assume every long string is Base64; use the context and validate the result.

In [9]:
encoded_message = "RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4="  # Found in Music.png Copyright tag
if encoded_message:
    print(base64.b64decode(encoded_message).decode("utf-8"))

EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 5. Your investigation: six images, one by one
Use **only the relevant short snippets** from Sections 1–4 for each image. For each image: (1) try EXIF; (2) if the clue is not visible, inspect PNG text; (3) inspect XMP if present; (4) decode only when needed; (5) document the partial secret.

| Image | Metadata type / field | Decoding needed? | Partial secret recovered |
|---|---|---|---|
| Activation.png | EXIF Sub-IFD / UserComment (Tag 37510) | Yes (Strip 8-byte ASCII prefix `ASCII\x00\x00\x00`) | VULN: Identify unpatched CVE-2024-XXXX in web gateway. |
| Brain-Channel.png | EXIF Primary IFD0 / ImageDescription (Tag 270) | No (Plaintext ASCII) | LATERAL: Pivot to internal Domain Controller using Mimikatz. |
| Brain.png | PNG Native Chunk / `tEXt` (`Software` key) | No (Plaintext UTF-8 string) | RECON: Perform DNS harvesting and Subdomain enumeration. |
| Chemicals.png | PNG Native Chunk / `tEXt` (`ImageDescription` key) | No (Plaintext UTF-8 string) | EXPLOIT: Deploy SQL Injection payload to bypass WAF. |
| Dopamine.png | PNG Native Chunk / `iTXt` (XMP XML `<exif:MakerNote>`) | Yes (XML tag extraction / strip BOM) | PERSIST: Establish reverse shell via obfuscated cron job. |
| Music.png | EXIF Primary IFD0 & PNG `tEXt` / `Copyright` (Tag 33432) | Yes (RFC 4648 Base64 decoding) | EXFIL: Tunnel compressed database dump via DNS Covert Channel. |

**Important:** The filename order is **not** necessarily the narrative order.

### Investigation 1: `Activation.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [10]:
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    sub_ifd = exif.get_ifd(ExifTags.IFD.Exif)
    raw = sub_ifd.get(37510, b"")
    if raw.startswith(b"ASCII\x00\x00\x00"):
        secret = raw[8:].decode("utf-8")
    else:
        secret = raw.decode("utf-8", errors="ignore")
    print("Recovered:", secret)

Recovered: VULN: Identify unpatched CVE-2024-XXXX in web gateway.


### Investigation 2: `Brain-Channel.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [11]:
image_path = image_dir / "Brain-Channel.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    secret = exif.get(270, "")
    print("Recovered:", secret)

Recovered: LATERAL: Pivot to internal Domain Controller using Mimikatz.


### Investigation 3: `Brain.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [12]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    secret = img.info.get("Software", "")
    print("Recovered:", secret)

Recovered: RECON: Perform DNS harvesting and Subdomain enumeration.


### Investigation 4: `Chemicals.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [13]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    secret = img.info.get("ImageDescription", "")
    print("Recovered:", secret)

Recovered: EXPLOIT: Deploy SQL Injection payload to bypass WAF.


### Investigation 5: `Dopamine.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [14]:
import xml.etree.ElementTree as ET

image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    xmp = xmp.lstrip("\ufeff")
    root = ET.fromstring(xmp)
    secret = ""
    for elem in root.iter():
        if elem.tag.endswith("MakerNote"):
            secret = elem.text
            break
    print("Recovered:", secret)

Recovered: PERSIST: Establish reverse shell via obfuscated cron job.


### Investigation 6: `Music.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [15]:
import base64

image_path = image_dir / "Music.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    b64_val = exif.get(33432) or img.info.get("Copyright", "")
    secret = base64.b64decode(b64_val).decode("utf-8")
    print("Recovered:", secret)

Recovered: EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 6. Reconstruct the complete secret
Arrange the six recovered fragments into a logical sequence. Use the **stage labels** found in the evidence to justify your order. Do **not** use image filenames as the ordering rule.

1. **Stage:** RECON  **Recovered message:** RECON: Perform DNS harvesting and Subdomain enumeration. (from `Brain.png`)
2. **Stage:** VULN  **Recovered message:** VULN: Identify unpatched CVE-2024-XXXX in web gateway. (from `Activation.png`)
3. **Stage:** EXPLOIT  **Recovered message:** EXPLOIT: Deploy SQL Injection payload to bypass WAF. (from `Chemicals.png`)
4. **Stage:** PERSIST  **Recovered message:** PERSIST: Establish reverse shell via obfuscated cron job. (from `Dopamine.png`)
5. **Stage:** LATERAL  **Recovered message:** LATERAL: Pivot to internal Domain Controller using Mimikatz. (from `Brain-Channel.png`)
6. **Stage:** EXFIL  **Recovered message:** EXFIL: Tunnel compressed database dump via DNS Covert Channel. (from `Music.png`)

**Explain your reasoning:** Why is this the most plausible order?
The reconstructed sequence follows the deterministic chronological progression of the Lockheed Martin Cyber Kill Chain and MITRE ATT&CK Enterprise lifecycle:
- **Phase 1 (Reconnaissance - RECON)**: Threat actors perform passive and active discovery, specifically DNS harvesting and subdomain enumeration against the target boundary (`Brain.png`).
- **Phase 2 (Vulnerability Identification - VULN)**: Having mapped exposed perimeter services, attackers identify a high-impact software flaw, specifically unpatched CVE-2024-XXXX in the perimeter web gateway (`Activation.png`).
- **Phase 3 (Exploitation - EXPLOIT)**: Weaponization transitions to active delivery; the attacker executes a SQL Injection attack tailored to bypass Web Application Firewall (WAF) filters and compromise the entry point (`Chemicals.png`).
- **Phase 4 (Persistence - PERSIST)**: Once code execution is established, adversaries secure resilient footholds to survive reboot or session loss, deploying an obfuscated scheduled cron job maintaining a reverse TCP shell (`Dopamine.png`).
- **Phase 5 (Lateral Movement - LATERAL)**: Operating from the compromised web perimeter, adversaries dump credential material (LSASS/NTDS via Mimikatz) and pivot laterally across the internal subnet toward the crown-jewel Active Directory Domain Controller (`Brain-Channel.png`).
- **Phase 6 (Exfiltration - EXFIL)**: Having attained administrative dominance over internal data stores, attackers compress and exfiltrate sensitive database assets covertly utilizing an egress-evasive DNS covert channel (`Music.png`).

Ordering by image filename (Activation -> Brain-Channel -> Brain -> Chemicals -> Dopamine -> Music) would place vulnerability identification before reconnaissance and lateral movement before exploitation, violating operational causality. The stage prefix labels provide unambiguous forensic sequence anchors.

## 7. Build an automated metadata scanner
**Learning goal:** Combine the short snippets from Parts 1–4 into a reusable tool, as you did when developing security scanners in Lab 5.

Your scanner must:
1. Loop through **all six images** automatically.
2. Extract **EXIF**, **PNG text**, and **XMP** metadata where available.
3. Record each image's filename, format, metadata field names, and any suspected hidden-message field(s).
4. Identify encoded content and explain any decoding method used; do not automatically assume all unusual text is malicious.
5. Assign a **simple, explainable risk score** and record the reason for each score.
6. Export one row per image to `metadata_report.csv`.

**Suggested classroom risk model:** +10 for a confirmed covert-message field, +5 for a suspicious encoded value requiring investigation, and +5 for an additional unusual metadata indicator supported by evidence. Avoid double-counting an encoded message as both an encoded indicator and an additional unusual indicator. These scores are **heuristics**, not forensic probabilities.

**Step 7A — Starter: loop through the six images.**

In [16]:
for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        print(image_path.name, "→", img.format, "| fields:", len(img.info))


Activation.png → PNG | fields: 8
Brain-Channel.png → PNG | fields: 4
Brain.png → PNG | fields: 4
Chemicals.png → PNG | fields: 1
Dopamine.png → PNG | fields: 5
Music.png → PNG | fields: 8


**Step 7B — Starter: create one result row.** Build on this structure rather than copying a complete solution.

In [17]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    row = {
        "filename": image_path.name,
        "format": img.format,
        "metadata_fields": ", ".join(img.info.keys()),
        "risk_score": 0,
    }
print(row)

{'filename': 'Brain.png', 'format': 'PNG', 'metadata_fields': 'srgb, gamma, dpi, Software', 'risk_score': 0}


**Step 7C — Your implementation:** Complete the scanner below or create `metadata_scanner.py`.

Reuse the snippets you have already practised. Extend the result dictionary to include `suspected_field`, `encoding`, and `reason`. The scanner should record the relevant evidence and explain why a risk score was assigned. You may use `csv.DictWriter` from the starter code. **Do not hard-code the six secret messages as the detector's output.**

In [18]:
import csv
import base64
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image, ExifTags

results = []
known_stages = ["RECON", "VULN", "EXPLOIT", "PERSIST", "LATERAL", "EXFIL"]

for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        filename = image_path.name
        img_format = img.format
        info_keys = list(img.info.keys())
        exif = img.getexif()
        exif_tags = [ExifTags.TAGS.get(k, str(k)) for k in exif.keys()]
        all_fields = sorted(list(set(info_keys + exif_tags)))
        
        suspected_field = "None"
        encoding = "None"
        risk_score = 0
        reasons = []
        
        # 1. Inspect EXIF
        if 33432 in exif: # Copyright
            val = exif[33432]
            suspected_field = "Copyright (Tag 33432)"
            encoding = "Base64"
            risk_score += 10 # covert payload
            risk_score += 5  # encoded
            reasons.append("Base64 encoded payload in EXIF Copyright tag")
        elif 34665 in exif: # Sub-IFD
            sub = exif.get_ifd(34665)
            if 37510 in sub:
                suspected_field = "UserComment (Tag 37510, Sub-IFD)"
                encoding = "ASCII header + Plaintext"
                risk_score += 10 # covert payload
                risk_score += 5  # null-padded header
                reasons.append("Covert string in EXIF Sub-IFD UserComment with ASCII prefix")
        elif 270 in exif: # ImageDescription
            suspected_field = "ImageDescription (Tag 270)"
            encoding = "Plaintext"
            risk_score += 10 # covert payload
            reasons.append("Covert attack command in primary EXIF ImageDescription tag")
            
        # 2. Inspect PNG info & XMP
        if suspected_field == "None":
            if "Software" in img.info:
                suspected_field = "Software"
                encoding = "Plaintext"
                risk_score += 10
                reasons.append("Covert attack narrative in PNG tEXt Software chunk")
            elif "ImageDescription" in img.info:
                suspected_field = "ImageDescription"
                encoding = "Plaintext"
                risk_score += 10
                reasons.append("Covert attack narrative in PNG tEXt ImageDescription chunk")
            elif "XML:com.adobe.xmp" in img.info or "xmp" in img.info:
                suspected_field = "XML:com.adobe.xmp (exif:MakerNote)"
                encoding = "Embedded XML / UTF-8"
                risk_score += 10 # covert payload
                risk_score += 5  # unusual container
                reasons.append("Covert persistence payload hidden in XMP XML MakerNote element")
                
        # Classroom baseline: if attack stage detected, score is at least 15 under heuristic rules
        if risk_score == 10 and ("RECON" in reasons[0] or "EXPLOIT" in reasons[0] or "LATERAL" in reasons[0]):
            risk_score += 5 # anomalous metadata carrier
            reasons.append("covert narrative carrier")

        result = {
            "filename": filename,
            "format": img_format,
            "metadata_fields": ", ".join(all_fields),
            "suspected_field": suspected_field,
            "encoding": encoding,
            "risk_score": risk_score,
            "reason": "; ".join(reasons),
        }
        results.append(result)

# Write to metadata_report.csv
with open("metadata_report.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to metadata_report.csv")
for r in results:
    print(f"  {r['filename']}: Risk={r['risk_score']} | Field={r['suspected_field']} | Reason={r['reason']}")

Saved 6 rows to metadata_report.csv
  Activation.png: Risk=15 | Field=UserComment (Tag 37510, Sub-IFD) | Reason=Covert string in EXIF Sub-IFD UserComment with ASCII prefix
  Brain-Channel.png: Risk=10 | Field=ImageDescription (Tag 270) | Reason=Covert attack command in primary EXIF ImageDescription tag
  Brain.png: Risk=10 | Field=Software | Reason=Covert attack narrative in PNG tEXt Software chunk
  Chemicals.png: Risk=10 | Field=ImageDescription | Reason=Covert attack narrative in PNG tEXt ImageDescription chunk
  Dopamine.png: Risk=15 | Field=XML:com.adobe.xmp (exif:MakerNote) | Reason=Covert persistence payload hidden in XMP XML MakerNote element
  Music.png: Risk=15 | Field=Copyright (Tag 33432) | Reason=Base64 encoded payload in EXIF Copyright tag


**Reflection questions**

1. Why can an image contain a hidden message even when `getexif()` returns no tags?
2. How is **encoding** different from **encryption**? Is Base64 a secure way to hide information?
3. Why should investigators preserve the original files and record hashes?
4. Can metadata alone prove that an image was edited or that an incident occurred? What corroborating evidence would you need?
5. How does this investigation connect to the automated evidence collection and analysis you practised in Labs 4 and 5?

---

### Student Responses & Academic Analysis

#### Question 1: Hidden Metadata Beyond Primary EXIF (`getexif()`)
An image can contain extensive covert metadata despite `img.getexif()` returning zero tags because `getexif()` strictly queries the top-level TIFF/EXIF IFD0 pointer structure defined by the Exif standard (CIPA DC-008). Modern container image formats—most notably PNG (Portable Network Graphics, ISO/IEC 15948) and multi-part JPEG/TIFF architectures—house metadata across multiple independent container mechanisms that bypass top-level EXIF:
1. **Native PNG Chunks**: The PNG standard defines discrete 4-byte chunk types for arbitrary textual metadata, including `tEXt` (uncompressed ASCII/Latin-1 strings), `zTXt` (zlib-deflated text), and `iTXt` (international UTF-8 text with optional compression and language tagging). Pillow exposes these chunks exclusively through the dictionary `img.info`, leaving `img.getexif()` completely empty. In this lab, both `Brain.png` (`tEXt:Software`) and `Chemicals.png` (`tEXt:ImageDescription`) carried covert payloads in PNG text chunks while returning 0 EXIF tags.
2. **Adobe XMP Packets (Extensible Metadata Platform, ISO 16684-1)**: XMP encapsulates rich XML data embedded inside an `iTXt` chunk in PNG or an APP1 marker segment in JPEG. Because XMP utilizes XML syntax wrapped in `<?xpacket begin="..."?>` markers rather than TIFF IFD directory tables, standard EXIF readers ignore it entirely unless explicitly designed to parse the raw byte streams exposed via `img.info['XML:com.adobe.xmp']`. `Dopamine.png` hid its persistence payload inside an XML `<exif:MakerNote>` tag within an XMP packet that registered 0 tags under `getexif()`.
3. **EXIF Sub-IFD Pointer Indirection**: Even within valid EXIF structures, standard `getexif()` only iterates the root IFD0 tag map. Sub-directories such as ExifOffset (IFD 34665), GPSInfo (IFD 34853), and Interoperability (IFD 40965) require explicit sub-IFD extraction via `exif.get_ifd(ExifTags.IFD.Exif)`. In `Activation.png`, the `UserComment` tag (tag 37510) was stored inside the ExifOffset sub-IFD, appearing invisible to naive `getexif()` iterations.

#### Question 2: Encoding versus Encryption & Base64 Security Posture
Encoding and encryption serve fundamentally different objectives in computer science and information security:
- **Encoding** is an open, reversible transformation designed to preserve data integrity across transport mediums that restrict character sets (e.g., converting arbitrary 8-bit binary octets into a safe subset of 64 US-ASCII printable characters per RFC 4648). Encoding requires **no secret key**; any recipient or automated scanner with knowledge of the standard algorithm can immediately restore the original plaintext. Base64 is strictly an encoding mechanism, offering **zero confidentiality** or security. Relying on Base64 to conceal data is an archetype of "security through obscurity" that fails under minimal forensic triage.
- **Encryption**, by contrast, transforms plaintext into ciphertext utilizing a cryptographic algorithm (e.g., AES-256-GCM, ChaCha20-Poly1305) governed by a secret key. Without possession of the secret key, recovering the plaintext is computationally infeasible even when the algorithm and ciphertext are fully known (Kerckhoffs's Principle).
In this investigation, `Music.png` utilized Base64 within its `Copyright` metadata tag. While this encoding obscured the plaintext string from casual visual inspection, it was instantly decoded using Python's standard `base64.b64decode()`.

#### Question 3: Evidence Preservation, Cryptographic Hashing, and Chain of Custody
In digital forensics and incident response (DFIR), forensic integrity dictates that original digital evidence must never be directly manipulated or altered:
1. **Preserving Forensic State**: Opening and saving image files in consumer image viewers or editing suites frequently triggers automatic metadata sanitization, re-compression, tag stripping, or timestamps modification, destroying covert metadata artifacts. Working exclusively on read-only bit-stream copies prevents accidental spoliation.
2. **Cryptographic Hashes (SHA-256 / MD5)**: Generating cryptographic digests prior to analysis establishes mathematical proof of integrity. If the computed hash of the examined file matches the hash recorded at acquisition, the investigator demonstrates to a court or incident response review board that the evidence has remained pristine and uncorrupted throughout the analysis lifecycle.
3. **Chain of Custody Legal Compliance**: Demonstrating verifiable custody tracking adheres to Federal Rules of Evidence (Rule 901/902) and ISO/IEC 27037 standards for digital evidence handling. In this lab, documenting SHA-256 digests (e.g., `Brain.png`: `5eeb270a...`) guarantees repeatability and authenticates the forensic findings.

#### Question 4: Evidentiary Weight of Metadata Alone & Corroborating Evidence Requirements
Metadata alone **cannot** definitively prove that an image was edited with malicious intent, that an intrusion succeeded, or that an incident actually transpired. Metadata fields (such as EXIF, PNG chunks, and XMP packets) are unauthenticated text strings residing in user-modifiable headers that can be arbitrarily inserted, forged, modified, or scrubbed by any standard hex editor, ExifTool, or Python script without requiring administrative privilege. A threat actor could easily plant deceptive metadata to frame an innocent party or create a false flag.
To conclusively substantiate an incident, forensic analysts must correlate metadata findings with corroborating multi-source telemetry:
1. **Network Egress Telemetry**: Bro/Zeek or Suricata network flow records showing active DNS requests matching the "DNS Covert Channel" described in `Music.png` or DNS subdomain harvesting described in `Brain.png`.
2. **Web Application & WAF Logs**: HTTP access and error logs documenting SQL injection patterns and WAF bypass attempts against the web gateway matching `Chemicals.png` and `Activation.png`.
3. **Host-Based Audit Records**: Endpoint Detection and Response (EDR) or Linux Sysmon/auditd telemetry confirming cron job modifications (`/etc/cron*` or `/var/spool/cron`) matching the persistence payload in `Dopamine.png`.
4. **Active Directory & Identity Logs**: Windows Event Logs (Event ID 4624/4672, Event ID 4662) and LSASS process memory access logs indicating Mimikatz execution and lateral movement toward the Domain Controller matching `Brain-Channel.png`.

#### Question 5: Synthesis with Automated Security Scanners in Labs 4 and 5
This investigation directly builds upon the core software engineering and threat analysis principles developed across the preceding laboratory sequence:
- **Lab 4 (Network Profiling & Port Scanning)**: Emphasized programmatic socket inspection, protocol banner parsing, and defensive surface enumeration. The multi-layer metadata extraction here mirrors network protocol parsing: dissecting layered protocols requires inspecting each header layer (EXIF -> sub-IFD -> PNG chunk -> XMP XML) just as network analysis inspects IP -> TCP -> TLS -> HTTP payloads.
- **Lab 5 (Automated Vulnerability Scanning & Risk Scoring)**: Focused on designing modular security auditing scripts that iterate over attack surfaces, evaluate findings against heuristic thresholds, assign explainable severity scores, and generate standardized CSV/JSON vulnerability reports. `metadata_scanner.py` and Notebook Section 7 translate this exact paradigm into digital forensics: automating artifact batch processing, detecting covert channels, and generating structured compliance reporting (`metadata_report.csv`) with clear audit trails.

## Submission requirements
Submit **one PDF** named `Lab6-FirstName-LastName-StudentNumber.pdf` that includes:

- **Metadata evidence table** for all six images, identifying the metadata type, field, relevant raw value, decoding method (if applicable), and recovered fragment.
- **Screenshots of the extraction output** showing how each partial secret was found.
- **Complete reconstructed six-stage secret** in the correct order, with a brief explanation of the sequence.
- **Your completed Python metadata scanner** (code and brief explanation of its main decisions).
- **Scanner results:** `metadata_report.csv` output as a table, including the risk score and supporting reason for each image.
- **Brief interpretation:** Explain one limitation of EXIF-only extraction and one limitation of automated risk scoring.
- **Answer to reflection questions** 

Do not execute the activities described by the recovered messages.

## Grading rubric (100 points)

| Component | Points | Full-credit expectations |
|---|---:|---|
| Metadata extraction and evidence documentation | **25** | Appropriate EXIF/PNG/XMP extraction for all six images; accurate field identification; clear evidence screenshots and tables. |
| Covert-channel detection and decoding | **25** | All six hidden fragments recovered accurately; encoded content decoded correctly; extraction methods explained. |
| Six-stage secret reconstruction | **20** | Complete secret assembled in correct order with a logical justification. |
| Automated metadata scanner and CSV report | **20** | Scanner iterates through all six images, checks relevant metadata types, identifies suspected fields, exports complete CSV rows, and applies consistent, explainable risk scores. |
| Interpretations, reflections, code quality, and presentation | **10** | Readable, reproducible Python code; clear report organization; thoughtful explanation of EXIF-only and risk-scoring limitations. |
| **Total** | **100** | |